# Full-data analysis (all ~14M rows)

Notebooks 01-06 were developed on a 1M-row sample. This notebook reruns the analysis on the full dataset. The main gain is precision: the control group grows from ~150K to ~2.1M users, and the control group is what bounds precision (not the 12M treated).

Part 1 (this section): loading, covariate balance, ATE, MDE. Later parts (CATE models, deciles, policy curve) get added one step at a time.

Data note: the dataset was non-uniformly subsampled by Criteo for privacy, so absolute effect sizes are not Criteo's real business numbers.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from uplift.data import load_full_cached
from uplift.balance import smd, FEATURES
from uplift.ate import compute_ate, compute_mde

df = load_full_cached()   # first call downloads (~2 min) and caches to data/raw/ as parquet
print(df.shape, f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df[["treatment", "exposure", "visit", "conversion"]].mean()

## Covariate balance at scale: the 0.1 rule is too lenient

`|SMD| < 0.1` is a rule of thumb built for small samples. Under true randomization the SMD of each feature is centered on 0 with standard error roughly `sqrt(1/n_treated + 1/n_control)`. At 14M rows that SE is under 0.001, so an SMD of 0.05 is dozens of standard errors from zero - not sampling noise, even though it passes the 0.1 rule.

In [ ]:
n1, n0 = (df["treatment"] == 1).sum(), (df["treatment"] == 0).sum()
se_smd = np.sqrt(1 / n1 + 1 / n0)

balance = smd(df).sort_values(key=abs, ascending=False).to_frame("smd")
balance["z (smd / expected SE)"] = balance["smd"] / se_smd
print(f"expected SE of an SMD under true randomization: {se_smd:.5f}")
balance

Balance tables can't say whether a tiny imbalance *matters*. A more direct check: can the 12 features **predict treatment assignment**? Under true randomization the best possible AUC is 0.5.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

sub = df.sample(3_000_000, random_state=1)
tr, te = train_test_split(sub, test_size=0.3, random_state=0)

for name, model in [("logistic", LogisticRegression(max_iter=300)),
                    ("gradient boosting", HistGradientBoostingClassifier(max_iter=100))]:
    model.fit(tr[FEATURES], tr["treatment"])
    auc = roc_auc_score(te["treatment"], model.predict_proba(te[FEATURES])[:, 1])
    print(f"{name}: AUC predicting treatment from features = {auc:.4f}")

**Reading this:** SMDs up to ~0.05 are about 65 expected-SEs from zero, and the features predict assignment with AUC ~0.51 - statistically distinguishable from 0.5 but a very weak signal. So the assignment is *not* perfectly independent of the features, but the imbalance is tiny (features shift by ~5% of a standard deviation at most, and predict treatment barely better than a coin flip). Notebook 01 concluded "randomization held" from the 0.1 rule alone; at this scale the right statement is "very nearly random, with a small but real imbalance." Whether it biases the ATE depends on how strongly those features predict the outcomes - a covariate-adjusted ATE is the natural check (next step).

I don't have an explanation for the source of the imbalance from the data alone (features are anonymized); the dataset's own documentation and the privacy subsampling are possible places to look.

## ATE and MDE on the full data

In [ ]:
results = {o: compute_ate(df, o) for o in ["visit", "conversion"]}
rows = []
for o, r in results.items():
    m = compute_mde(r)
    rows.append({"outcome": o, "rate_treated": r.rate_treated, "rate_control": r.rate_control,
                 "ATE": r.ate, "ci_low": r.ci_low, "ci_high": r.ci_high,
                 "relative_lift": r.relative_lift, "MDE_abs": m.mde_abs, "MDE_relative": m.mde_relative})
pd.DataFrame(rows).set_index("outcome").T

## Compared with the 1M sample

| | 1M sample | full data |
|---|---|---|
| `visit` ATE | 0.01081 (CI 0.00974 to 0.01188) | 0.01034 (CI 0.01006 to 0.01063) |
| `visit` relative lift | 28.6% | 27.1% |
| `visit` MDE (relative) | ~4.1% | ~1.1% |
| `conversion` ATE | 0.00111 (CI 0.00085 to 0.00137) | 0.00115 (CI 0.00108 to 0.00122) |
| `conversion` relative lift | 55.3% | 59.5% |
| `conversion` MDE (relative) | ~18.2% | ~5.0% |

Point estimates barely move (the sample was already an unbiased draw), but the conversion CI is about 4x narrower. Both effects are unambiguous: each is many multiples of its MDE, and neither CI is close to zero.

**Caveat carried over from the balance check:** these are raw differences in means, which are unbiased only if assignment is independent of the outcomes' determinants. The small feature imbalance above is a reason to confirm with a covariate-adjusted estimate before treating these as final.

## Does the imbalance move the ATE? Covariate-adjusted estimates

The raw difference in means is unbiased only if assignment is independent of what drives the outcomes. Two checks that adjust for the 12 features:

1. **Linear adjustment (Lin-style):** regress the outcome on treatment, centered features, and treatment x feature interactions. The treatment coefficient is the adjusted ATE. Robust (HC0) standard errors.
2. **Cross-fitted doubly robust ATE:** flexible gradient-boosted outcome models (one per arm) plus an *estimated* propensity `e(x)`, in 2 folds so no row is scored by a model that saw it. Run on a 4M-row subsample for speed.

In [ ]:
X = df[FEATURES].to_numpy(np.float64).copy()
X = X - X.mean(axis=0)
t = df["treatment"].to_numpy(np.float64)
Z = np.column_stack([np.ones(len(df)), t, X, X * t[:, None]])
ZtZ_inv = np.linalg.inv(Z.T @ Z)

rows = []
for outcome in ["visit", "conversion"]:
    y = df[outcome].to_numpy(np.float64)
    beta = ZtZ_inv @ (Z.T @ y)
    resid = y - Z @ beta
    meat = (Z * resid[:, None]).T @ (Z * resid[:, None])
    se_adj = np.sqrt((ZtZ_inv @ meat @ ZtZ_inv)[1, 1])
    raw = results[outcome]
    rows.append({"outcome": outcome, "raw_ate": raw.ate, "adjusted_ate": beta[1], "adjusted_se": se_adj,
                 "diff": beta[1] - raw.ate, "diff / raw_se": (beta[1] - raw.ate) / raw.se})

del Z, X
pd.DataFrame(rows).set_index("outcome")

In [ ]:
sub = df.sample(4_000_000, random_state=7).reset_index(drop=True)
Xs, ts = sub[FEATURES].to_numpy(), sub["treatment"].to_numpy()
fold = np.arange(len(sub)) % 2

e = np.zeros(len(sub))
for k in (0, 1):
    prop = HistGradientBoostingClassifier(max_iter=100).fit(Xs[fold != k], ts[fold != k])
    e[fold == k] = prop.predict_proba(Xs[fold == k])[:, 1]
print(f"estimated propensity: min={e.min():.3f}, max={e.max():.3f}, sd={e.std():.4f}")

rows = []
for outcome in ["visit", "conversion"]:
    y = sub[outcome].to_numpy()
    phi = np.zeros(len(sub))
    for k in (0, 1):
        tr, te = fold != k, fold == k
        m1 = HistGradientBoostingClassifier(max_iter=100).fit(Xs[tr & (ts == 1)], y[tr & (ts == 1)])
        m0 = HistGradientBoostingClassifier(max_iter=100).fit(Xs[tr & (ts == 0)], y[tr & (ts == 0)])
        p1, p0 = m1.predict_proba(Xs[te])[:, 1], m0.predict_proba(Xs[te])[:, 1]
        phi[te] = (p1 - p0
                   + ts[te] * (y[te] - p1) / e[te]
                   - (1 - ts[te]) * (y[te] - p0) / (1 - e[te]))
    raw_sub = y[ts == 1].mean() - y[ts == 0].mean()
    rows.append({"outcome": outcome, "raw_ate (same subsample)": raw_sub,
                 "dr_ate": phi.mean(), "dr_se": phi.std() / np.sqrt(len(phi))})
pd.DataFrame(rows).set_index("outcome")

## What the adjustment shows

| outcome | raw ATE | linear-adjusted | cross-fitted DR (4M subsample; raw on same rows) |
|---|---|---|---|
| `visit` | 0.01034 | 0.00773 (SE 0.00013) | 0.00735 (SE 0.00024); raw 0.01008 |
| `conversion` | 0.00115 | 0.00100 (SE 0.00003) | 0.00111 (SE 0.00007); raw 0.00114 |

- **`visit`: the adjustment matters.** Both adjusted estimates land near 0.0074-0.0077, about 25% below the raw 0.0103 (a gap of ~18 raw standard errors). A ~0.05 SD imbalance is enough to move the ATE this much because the features strongly predict `visit`.
- **`conversion`: inconclusive.** The linear adjustment is ~13% lower, but the DR estimate is essentially the raw number. There is no convincing evidence of bias here.

**How to read this - do not just swap in the adjusted number.** Adjustment is valid only if the features capture whatever made assignment non-random *and* they are pre-treatment. If some features were measured after assignment, adjusting for them conditions on a post-treatment variable and could make the estimate *worse*. If the imbalance comes from the non-uniform privacy subsampling instead, both raw and adjusted may be off. Features are anonymized, so I can't tell these apart from the data. Honest reporting: the `visit` ATE is somewhere between ~0.0074 (adjusted) and ~0.0103 (raw), i.e. roughly a 20-27% relative lift, and the source of the discrepancy is unresolved.

Also flagged for the CATE work: `fit_dr_learner` in `cate_models.py` assumes a constant propensity of 0.85, which this notebook shows isn't exactly right. Next step is changing it to use an estimated propensity before the CATE sections are rerun on full data.

# Part 2: CATE estimation on the full data

Split 60/40 into **train** (~8.4M rows, models are fit here) and **eval** (~5.6M rows, held out; everything below is measured here). S, T, X and DR learners use a gradient-boosted base learner. The DR- and X-learners use an *estimated* propensity `e(x)` (the constant-0.85 shortcut was dropped after the imbalance check above), and the DR-learner cross-fits its nuisance models.

The causal forest is **not** run here: `econml`'s `CausalForestDML` does not scale to millions of rows, and running it on a subsample would not be comparable to the other models.

Fitting takes roughly 10 minutes; predictions are cached to `data/raw/full_cates.pkl` (gitignored) and reused on later runs.

In [ ]:
import pickle
import time
import matplotlib.pyplot as plt

from uplift.data import FULL_PATH
from uplift.cate_models import (
    fit_s_learner, predict_s_learner, fit_t_learner, predict_t_learner,
    fit_x_learner, predict_x_learner, fit_dr_learner, predict_dr_learner,
)
from uplift.evaluation import decile_table, qini_curve, auuc

CACHE = FULL_PATH.parent / "full_cates.pkl"
LEARNERS = {
    "S-learner": (fit_s_learner, predict_s_learner),
    "T-learner": (fit_t_learner, predict_t_learner),
    "X-learner": (fit_x_learner, predict_x_learner),
    "DR-learner": (fit_dr_learner, predict_dr_learner),
}

train, eval_ = train_test_split(df, test_size=0.4, stratify=df["treatment"], random_state=0)
print("train:", train.shape, "eval:", eval_.shape)

if CACHE.exists():
    cates = pickle.load(open(CACHE, "rb"))
    assert (cates["eval_index"] == eval_.index.to_numpy()).all(), "cache was built from a different split"
else:
    cates = {"eval_index": eval_.index.to_numpy()}
    for outcome in ["visit", "conversion"]:
        cates[outcome] = {}
        for name, (fit, predict) in LEARNERS.items():
            t0 = time.time()
            cates[outcome][name] = predict(fit(train, outcome), eval_)
            print(f"{outcome} {name}: {time.time() - t0:.0f}s")
    pickle.dump(cates, open(CACHE, "wb"))

## Sanity check and model summary

Held-out means, a Qini/AUUC score, and the measured ATE in the top and bottom predicted-CATE decile (99.5% Bonferroni CIs). `n_unique` counts distinct predicted values: a model that predicts only a few values can't rank users.

In [ ]:
rows = {}
for outcome in ["visit", "conversion"]:
    rows[outcome] = []
    for name, cate in cates[outcome].items():
        q = qini_curve(eval_, cate, outcome)
        d = decile_table(eval_, cate, outcome, alpha=0.05 / 10)
        rows[outcome].append({
            "model": name, "mean_predicted_cate": cate.mean(), "n_unique": len(set(cate.round(6))),
            "AUUC": auuc(q),
            "top_decile_ate": d["actual_ate"].iloc[-1],
            "top_decile_ci": f"[{d['ci_low'].iloc[-1]:.4f}, {d['ci_high'].iloc[-1]:.4f}]",
            "bottom_decile_ate": d["actual_ate"].iloc[0],
            "bottom_decile_ci": f"[{d['ci_low'].iloc[0]:.4f}, {d['ci_high'].iloc[0]:.4f}]",
        })
    print(f"=== {outcome} (eval-set raw ATE = {compute_ate(eval_, outcome).ate:.5f}) ===")
    print(pd.DataFrame(rows[outcome]).round(5).to_string(index=False))
    print()

## Decile tables (DR-learner and X-learner)

In [ ]:
for outcome in ["visit", "conversion"]:
    for name in ["DR-learner", "X-learner"]:
        d = decile_table(eval_, cates[outcome][name], outcome, alpha=0.05 / 10)
        print(f"=== {outcome} / {name} ===")
        print(d[["decile", "n", "mean_predicted_cate", "actual_ate", "ci_low", "ci_high"]].round(5).to_string(index=False))
        print()

## Qini curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, outcome in zip(axes, ["visit", "conversion"]):
    for name, cate in cates[outcome].items():
        q = qini_curve(eval_, cate, outcome)
        ax.plot(q["fraction"], q["qini"], label=name)
    ax.plot(q["fraction"], q["random"], "k--", label="random targeting")
    ax.set_title(outcome)
    ax.set_xlabel("fraction of users targeted (highest predicted CATE first)")
    ax.set_ylabel("incremental outcomes (eval set)")
    ax.legend()
plt.tight_layout()
plt.show()

## What the full data shows

**Heterogeneity is real and concentrated at the top.**
- `visit`: the top predicted decile has a measured ATE of ~0.060-0.065 (CI ~0.060 to 0.069), about **6x** the overall 0.0106. Decile 8 is ~0.008, decile 7 ~0.003, and deciles 0-6 are all near 0.0003-0.001. That is a smooth, monotone climb at the top and near-flat elsewhere, not the noisy middle seen at 1M rows.
- `conversion`: the top decile is ~0.0078-0.0086 (~**7x** the 0.00116 overall ATE); deciles 1-8 sit at ~0.0001-0.0002 (several CIs include zero). Roughly 65-75% of the total measured effect comes from the top 10% of users (approximate: computed from decile ATEs x equal bin sizes).
- Because each decile now has ~559K users, these estimates are tight. The middle of the ranking is not "noisy heterogeneity", it is close to no effect.

**Model comparison (single train/eval split, no uncertainty on AUUC - read loosely).**
- `visit`: all four models are within ~7% of each other on AUUC (~20,300-21,750). The S-learner works fine here.
- `conversion`: the S-learner collapses (3 distinct predicted values, negative AUUC). DR (~2,350) and X (~2,230) are the best and essentially tied (their top-decile CIs overlap); the T-learner is clearly weaker (~1,410), consistent with its noise problem on a rare outcome with a small control group.

**Sleeping dogs.** No decile of any model has a CI entirely below zero, for either outcome. With ~559K users per decile this is much stronger evidence than at 1M rows, but still a statement about *deciles*: a small hurt subgroup inside a decile could be averaged away. Features are anonymized, so any such group could not be described anyway.

**Calibration caveat - do not skip.** For `visit`, every model's mean predicted CATE (0.0070-0.0075) is ~30% below the eval-set raw ATE (0.0106) - almost exactly the raw-vs-adjusted gap found above (~0.0074 adjusted). The models condition on the features, so they behave like the adjusted estimate, whereas the decile "actual_ate" values are raw treated-vs-control differences that inherit the imbalance. Ranking (which deciles are high vs low) is the trustworthy output; the *absolute level* of the decile effects, especially `visit`, is uncertain by that same ~25-30%, for the unresolved reason discussed in Part 1.

# Part 3: Policy curve on the full data

Same question as notebook 05: which fraction of users to target. Two changes from the 1M-sample version:

1. **No selection on the same data that measures profit.** The eval set (~5.6M) is split 50/50. The cutoff (top-k deciles) is chosen to maximize profit on half A, then that cutoff's profit is *measured on half B*. Repeated over 5 seeds.
2. **Two cost models**, because the answer turns out to depend on it. Assumed economics as before (not Criteo's real numbers): $50 per conversion, $0.01 per impression.
   - *Every targeted user costs $0.01* (as in notebook 05).
   - *Only exposed users cost $0.01*: in this dataset an eligible user only costs an impression if Criteo actually won the auction (`exposure=1`), about 3.6% of treated users. `exposure` is used here purely for **cost accounting**, never for the causal estimate.

Outcome: `conversion`. Profit = `$50 x (decile ATE x users) - cost`.

In [ ]:
from uplift.policy import policy_curve

outcome = "conversion"
e_overall = eval_.loc[eval_["treatment"] == 1, "exposure"].mean()
print(f"overall exposure rate among treated: {e_overall:.4f}")

for model in ["DR-learner", "X-learner"]:
    cate = cates[outcome][model]
    w = eval_.assign(cate=cate)
    w["decile"] = pd.qcut(w["cate"].rank(method="first"), 10, labels=False)
    exp_by_decile = w[w["treatment"] == 1].groupby("decile")["exposure"].mean()
    print(f"
=== {model} ===")
    print("exposure rate among treated, by predicted-CATE decile (0 -> 9):", exp_by_decile.round(3).tolist())

    for label, cost in [("cost on every targeted user ($0.01)", 0.01),
                        (f"cost on exposed users only ($0.01 x {e_overall:.3f})", 0.01 * e_overall)]:
        print(f"-- {label}")
        for seed in range(5):
            idx = np.arange(len(eval_))
            a, b = train_test_split(idx, test_size=0.5, stratify=eval_["treatment"], random_state=seed)
            curve_a = policy_curve(decile_table(eval_.iloc[a], cate[a], outcome), cost_per_impression=cost)
            curve_b = policy_curve(decile_table(eval_.iloc[b], cate[b], outcome), cost_per_impression=cost)
            k = curve_a["net_profit"].idxmax()
            print(f"   seed {seed}: cutoff chosen on A = top {k + 1} decile(s); profit on B at that cutoff = "
                  f"{curve_b['net_profit'].iloc[k]:,.0f} (B's own best: top {curve_b['net_profit'].idxmax() + 1}, "
                  f"{curve_b['net_profit'].max():,.0f})")
        full = policy_curve(decile_table(eval_, cate, outcome), cost_per_impression=cost)
        print("   full-eval net profit if you target the top 1, 2, ..., 10 deciles:")
        print("  ", full["net_profit"].round(0).astype(int).tolist())

## Result

**The top decile is where the value is.** Under "every targeted user costs $0.01", the DR-learner's profit from the top decile alone is ~$234K (eval set, assumed economics), and adding every remaining decile changes it by at most ~$2K (<1%). The curve is flat from the top decile to 100%: the other deciles' effects (~0.0001-0.0002) are worth about what an impression costs ($0.0002 breakeven).

**So the exact cutoff is not identified, and that is the honest finding.** The cutoff chosen on half A jumps between top 2, 5 and 10 deciles across seeds (DR) and between top 2 and 3 (X). It doesn't matter much - profit measured on half B is within a few percent of B's own best in each case - but it means "the optimal fraction is 0.7" style statements are not supportable.

**The answer flips with the cost model.**
- *Cost on every targeted user:* target roughly the top 1-3 deciles; everything beyond is a wash.
- *Cost on exposed users only:* impressions cost only when an auction is won, and the middle deciles are exposed only ~2% of the time, so they cost ~$0.0002 per eligible user against ~$0.0075 of benefit. Profit rises monotonically to targeting **everyone** (~$288K vs ~$234K for the top decile, DR).
- The exposure rate itself varies by decile (DR: ~2% in the middle, 15% in the top decile), which is also why the top decile is where the auction spends its money.

Which model is right depends on how the ad is actually bought and is not knowable from this data. The defensible statement: **the top ~10% of users yields most of the value under any cost model; whether to go beyond that depends on whether you pay per eligible user or per delivered impression.**

Caveats: dollar figures rest on assumed economics ($50 per conversion, $0.01 per impression) and are illustrative only; decile effects are raw treated-vs-control differences (the imbalance noted in Part 1 affects `visit` far more than `conversion`); and the data was non-uniformly subsampled, so absolute conversion counts are not Criteo's.